# 🦙 Aula 04 — llama.cpp direto (llama.app + Gemma 4 E2B)
## Guilda de IA — Introdução à IA Generativa

<a href="https://colab.research.google.com/github/luksamuk/guilda-ia/blob/main/notebooks/aula04_llamaapp_colab.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Notebook **experimental**: aqui a gente deixa o Ollama de lado e conversa com o motor *llama.cpp* direto — o mesmo que roda embaixo de tudo.

- Instalação via **llama.app** (distribuição oficial da equipe ggml)
- Modelo: **Gemma 4 E2B (Q4_0)** baixado direto do Hugging Face
- Mesma API OpenAI-compatible da apostila: `/v1/chat/completions`
- Zero API key, zero conta de provedor


## 1. Setup

⚠️ Recomendado: `Runtime → Change runtime type` → **T4 GPU** (o instalador detecta a GPU sozinho). CPU também roda, só mais devagar.


In [ ]:
# Instalar o llama (llama.app — distribuição oficial da equipe ggml)
!curl -LsSf https://llama.app/install.sh | sh

!~/.local/bin/llama version  # binario pronto pro PATH ~/.local/bin


In [ ]:
# Subir o servidor com o Gemma 4 E2B (baixa do Hugging Face no primeiro run, ~3 min)
import os, subprocess, requests, time

LLAMA = os.path.expanduser("~/.local/bin/llama")
PORT = 8080

subprocess.run(["pkill", "-f", "local/bin/llama"], capture_output=True)
time.sleep(1)

process = subprocess.Popen(
    [LLAMA, "serve",
     "--port", str(PORT),
     "-hf", "ggml-org/gemma-4-E2B-it-GGUF:Q4_0",
     "-c", "8192",
     "-rea", "off",
     "--temp", "0.7", "--top-k", "40", "--top-p", "0.95"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

print("⏳ Aguardando servidor + download do modelo...")
for i in range(300):
    try:
        if requests.get(f"http://localhost:{PORT}/health", timeout=2).status_code == 200:
            print(f"✅ Servidor pronto! ({i*2}s)")
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Servidor não subiu — rode a célula de novo ou confira o runtime")


## 2. Conversar com o modelo

⚠️ O download do modelo (~2.8 GB) acontece na primeira chamada em `-hf`. Paciência!


In [ ]:
import requests

URL = f"http://localhost:{PORT}/v1/chat/completions"

def chat(messages, max_tokens=300):
    payload = {
        "model": "gemma-4-E2B",  # o nome é livre — o servidor só tem um modelo
        "messages": messages,
        "max_tokens": max_tokens,
    }
    r = requests.post(URL, json=payload, timeout=300)
    r.raise_for_status()
    return r.json()

# Chat simples
resultado = chat([
    {"role": "system", "content": "Você é um assistente útil. Responda em português."},
    {"role": "user", "content": "Qual é a capital de Minas Gerais?"},
])
print(f"🤖 {resultado['choices'][0]['message']['content']}")
print(f"📊 Tokens: {resultado['usage']['completion_tokens']}")


In [ ]:
# Chat com memória: o histórico completo vai em toda chamada
historico = [
    {"role": "system", "content": "Você é um assistente útil. Responda em português. Seja conciso."},
]

historico.append({"role": "user", "content": "Meu nome é Lucas e eu amo IA."})
r1 = chat(historico, max_tokens=150)
resposta1 = r1["choices"][0]["message"]["content"]
historico.append({"role": "assistant", "content": resposta1})
print(f"🤖: {resposta1}")

historico.append({"role": "user", "content": "Qual é o meu nome?"})
r2 = chat(historico, max_tokens=150)
resposta2 = r2["choices"][0]["message"]["content"]
historico.append({"role": "assistant", "content": resposta2})
print(f"🤖: {resposta2}")  # Deve lembrar!


## 3. Bônus: E o tal do raciocínio?

Alguns modelos (como o LFM2.5 da apostila) **pensam antes de responder** — o rascunho vai no campo `reasoning_content` e consome o orçamento de `max_tokens`.

Foi por isso que subimos o servidor com **`-rea off`**: desliga o thinking do Gemma e a resposta vem inteira no `content`.

Teste a diferença: rode a célula acima num servidor **sem** `-rea off` e compare as chaves da resposta!


---

✅ Mesma API do Ollama, mesmo payload — só muda a URL. O dialeto é sempre `/v1/chat/completions`.

*Material da Guilda de IA — UFVJM 2026.2*  
*Notebook experimental — llama.app (ggml)*
